In [74]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
import networkx as nx
import matplotlib.pyplot as plt
import copy
import time
import torch
import math
from typing import List, Tuple, Optional
from collections import deque
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
embedding_path = "/home/hieunt/verl/data/embedding_data/embeddings_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy"
pairwise_path = "/home/hieunt/verl/data/embedding_data/pairwise_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398_matrix.npy"
embeddings = np.load(embedding_path)
pairwise_dists = np.load(pairwise_path)
embeddings.shape

(17398, 1024)

In [75]:
def partition_equal_clusters(
    X: np.ndarray,
    n_clusters: int,
    *,
    random_state: int = 42,
    n_init: int = 10,
    max_iter: int = 300,
) -> list[np.ndarray]:
    """
    Partition embeddings into n_clusters with *equal* cluster sizes.

    Parameters
    ----------
    X : (n_samples, n_features) array
    n_clusters : int
        Must divide n_samples exactly.

    Returns
    -------
    clusters : list[np.ndarray]
        Length = n_clusters; each is an index array with identical size.
    """
    n, d = X.shape
    if n_clusters <= 0:
        raise ValueError("n_clusters must be positive.")
    if n % n_clusters != 0:
        raise ValueError(
            f"n_samples={n} not divisible by n_clusters={n_clusters}. "
            "Pick a divisor of n_samples for equal-sized clusters."
        )
    target = n // n_clusters

    # 1) KMeans for centroids
    km = KMeans(
        n_clusters=n_clusters,
        n_init=n_init,
        max_iter=max_iter,
        random_state=random_state,
        verbose=0,
    ).fit(X)
    C = km.cluster_centers_  # (k, d)

    # 2) Squared distances to each centroid
    X2 = np.sum(X * X, axis=1, keepdims=True)        # (n, 1)
    C2 = np.sum(C * C, axis=1, keepdims=True).T      # (1, k)
    XC = X @ C.T                                     # (n, k)
    dists = X2 + C2 - 2.0 * XC                       # (n, k)
    # Guard against tiny negative due to float error
    dists = np.maximum(dists, 0.0)

    # 3) Preferences (nearest first)
    prefs = np.argsort(dists, axis=1)  # (n, k)

    # 4) Capacity-constrained assignment (fixed: don't reassign in-round)
    quota = np.full(n_clusters, target, dtype=int)
    assigned = np.full(n, -1, dtype=int)

    for r in range(n_clusters):
        remaining_mask = (assigned == -1)
        if not remaining_mask.any():
            break
        idxs = np.where(remaining_mask)[0]
        desired_clusters = prefs[remaining_mask, r]  # r-th choice for each remaining point

        for c in range(n_clusters):
            if quota[c] <= 0:
                continue
            cand = idxs[desired_clusters == c]
            if cand.size == 0:
                continue
            # **Fix**: filter out any that may have been assigned earlier in this same round
            cand = cand[assigned[cand] == -1]
            if cand.size == 0:
                continue

            # take closest up to remaining quota
            take_count = min(quota[c], cand.size)
            order = np.argsort(dists[cand, c])[:take_count]
            take = cand[order]

            assigned[take] = c
            quota[c] -= take.size

    # 5) Fallback for any leftovers (should be rare)
    if (assigned == -1).any():
        for i in np.where(assigned == -1)[0]:
            for c in prefs[i]:
                if quota[c] > 0:
                    assigned[i] = c
                    quota[c] -= 1
                    break

    clusters = [np.where(assigned == c)[0] for c in range(n_clusters)]

    # Sanity checks
    assert sum(len(g) for g in clusters) == n
    assert all(len(g) == target for g in clusters), "Internal error: unequal sizes."
    return clusters


def mst_from_embeddings_full(X: np.ndarray, metric: str = "euclidean"):
    """
    Build the Euclidean MST of a set of embeddings.

    Parameters
    ----------
    X : (n, d) array
        Embeddings (float32 is fine to save memory).
    metric : str
        Any metric supported by scipy.spatial.distance.pdist.

    Returns
    -------
    edges : (n-1, 3) int/float array
        Each row is [u, v, w] meaning an MST edge u--v with weight w.
    total_weight : float
        Sum of weights in the MST.
    """
    # Pairwise distances (dense, O(n^2) memory)
    D = squareform(pdist(X, metric=metric)).astype(np.float32)

    # SciPy expects a sparse matrix (we can pass dense; it treats it as weighted graph)
    T = minimum_spanning_tree(D)        # returns a CSR sparse matrix with MST edges
    T = T.tocoo()

    # Collect edges (note: MST is directed in CSR; make it undirected by reading both directions if needed)
    edges = np.column_stack([T.row, T.col, T.data])
    total_weight = float(T.data.sum())
    return edges, total_weight

def _mst_edges_to_adj(edges: np.ndarray, n: int) -> List[List[Tuple[int, float]]]:
    """
    Convert MST edges [u, v, w] to an undirected adjacency list and
    sort neighbors by ascending weight so traversals stay local.
    """
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        u = int(u); v = int(v); w = float(w)
        adj[u].append((v, w))
        adj[v].append((u, w))
    for nbrs in adj:
        nbrs.sort(key=lambda x: x[1])   # prefer lighter edges first
    return adj

def _farthest_vertex(adj: List[List[Tuple[int, float]]], start: int) -> int:
    """
    On a tree, any DFS/BFS with accumulated weights yields the unique path cost.
    Returns the vertex farthest (by weighted distance) from 'start'.
    """
    n = len(adj)
    dist = np.full(n, np.inf)
    parent = [-1]*n
    dist[start] = 0.0
    q = deque([start])
    while q:
        u = q.popleft()
        for v, w in adj[u]:
            if parent[u] == v:  # don't go back to parent
                continue
            # unique simple path in a tree; the first time we set dist[v] is optimal
            if np.isinf(dist[v]):
                dist[v] = dist[u] + w
                parent[v] = u
                q.append(v)
    return int(np.nanargmax(dist))

def _tree_diameter_end(adj: List[List[Tuple[int, float]]]) -> int:
    """
    Heuristic for a good DFS root: pick one endpoint of the tree diameter.
    """
    u = _farthest_vertex(adj, 0)
    v = _farthest_vertex(adj, u)
    return v

def mst_dfs_order(edges: np.ndarray, n: int, root: Optional[int] = None) -> List[int]:
    """
    Produce a permutation of indices by DFS on the MST, preferring light edges first.
    Consecutive indices in the output tend to be close in the embedding space.
    """
    adj = _mst_edges_to_adj(edges, n)
    if root is None:
        root = _tree_diameter_end(adj)

    order: List[int] = []
    seen = [False] * n
    stack = [root]
    while stack:
        u = stack.pop()
        if seen[u]:
            continue
        seen[u] = True
        order.append(u)
        # neighbors already sorted light->heavy; push in reverse so light pops first
        for v, _w in reversed(adj[u]):
            if not seen[v]:
                stack.append(v)

    # Safety: if the graph somehow wasn’t connected (shouldn’t happen for an MST),
    # append any missed nodes.
    if len(order) < n:
        order.extend([i for i in range(n) if not seen[i]])
    return order

def evaluate_ordering(
    X: np.ndarray,
    order: List[int],
    metric: str = "euclidean",
    mst_edges: Optional[np.ndarray] = None,
    mst_total_weight: Optional[float] = None,
) -> dict:

    order = np.asarray(order, dtype=int)
    Xo = X[order]

    # Fast path for Euclidean
    if metric == "euclidean":
        d = np.linalg.norm(Xo[1:] - Xo[:-1], axis=1)
    else:
        # Generic metric: compute per-pair distances
        d = np.array([
            float(cdist(Xo[i:i+1], Xo[i+1:i+2], metric=metric)[0, 0])
            for i in range(len(Xo) - 1)
        ])

    out = {
        "mean_consecutive_distance": float(d.mean()),
        "median_consecutive_distance": float(np.median(d)),
        "max_consecutive_distance": float(d.max()),
        "total_path_length": float(d.sum()),
    }

    if mst_edges is not None:
        # Count how many consecutive pairs are also MST edges (undirected)
        mst_edge_set = { (min(int(u), int(v)), max(int(u), int(v))) for u, v, _ in mst_edges }
        hits = 0
        for a, b in zip(order[:-1], order[1:]):
            key = (min(int(a), int(b)), max(int(a), int(b)))
            hits += key in mst_edge_set
        out["frac_consecutive_pairs_in_mst"] = hits / (len(order) - 1)

    if mst_total_weight is not None and (len(Xo) > 1):
        # Compare the path’s total consecutive distance to the MST weight.
        # (A path is a special tree, so this ratio is typically >= 1.)
        out["path_vs_mst_ratio"] = float(out["total_path_length"] / mst_total_weight)

    return out

In [76]:
clusters = partition_equal_clusters(embeddings[:17152], n_clusters=8)
print([len(c) for c in clusters])
print([len(set(c)) for c in clusters])
print(len(set.union(*[set(c) for c in clusters])))

data_indices = None
for c in clusters:
    cluster_embedding = embeddings[c]
    mst_edges, mst_total_weight = mst_from_embeddings_full(cluster_embedding, metric="euclidean")
    order = mst_dfs_order(mst_edges, len(cluster_embedding))
    report = evaluate_ordering(
        cluster_embedding,
        order,
        mst_edges=mst_edges,
        mst_total_weight=mst_total_weight,
    )
    print(report)
    cluster_indices = np.array(c)[order].reshape(32, -1)
    if data_indices is None:
        data_indices = cluster_indices
    else:
        data_indices = np.vstack([data_indices, cluster_indices])
    print(data_indices.shape)

batches = []
for i in range(data_indices.shape[1]):
    batches.append(data_indices[:, i].tolist())

indices_list = [idx for batch in batches for idx in batch] + list(range(17152, 17398))
# save to file
np.save("/home/hieunt/verl/data/organised_indices/kmeans_mst_indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy", np.array(indices_list))


[2144, 2144, 2144, 2144, 2144, 2144, 2144, 2144]
[2144, 2144, 2144, 2144, 2144, 2144, 2144, 2144]
17152
{'mean_consecutive_distance': 0.5995182394981384, 'median_consecutive_distance': 0.6611638069152832, 'max_consecutive_distance': 1.0867162942886353, 'total_path_length': 1284.767578125, 'frac_consecutive_pairs_in_mst': 0.5426971535230984, 'path_vs_mst_ratio': 1.138335382493264}
(32, 67)
{'mean_consecutive_distance': 0.6972630023956299, 'median_consecutive_distance': 0.7774720788002014, 'max_consecutive_distance': 1.1600221395492554, 'total_path_length': 1494.234619140625, 'frac_consecutive_pairs_in_mst': 0.5310312645823612, 'path_vs_mst_ratio': 1.1207561276998883}
(64, 67)
{'mean_consecutive_distance': 0.6762315034866333, 'median_consecutive_distance': 0.7262173891067505, 'max_consecutive_distance': 1.113965630531311, 'total_path_length': 1449.1640625, 'frac_consecutive_pairs_in_mst': 0.5324311712552496, 'path_vs_mst_ratio': 1.142805131223744}
(96, 67)
{'mean_consecutive_distance': 0

In [77]:
dists = []
for j in range(len(batches) - 1):
    for i in range(256):
        d = np.linalg.norm(embeddings[batches[j][i]] - embeddings[batches[j + 1]], axis=1)
        dists.append(d.min())
print(sum(dists) / len(dists))

0.63712186


In [83]:
def map_inference_dpp_greedy(kernel_matrix, max_length, epsilon=1E-10):
    """
    Our proposed fast implementation of the greedy algorithm
    :param kernel_matrix: 2-d array
    :param max_length: positive int
    :param epsilon: small positive scalar
    :return: list
    """
    item_size = kernel_matrix.shape[0]
    cis = np.zeros((max_length, item_size))
    di2s = np.copy(np.diag(kernel_matrix))
    selected_items = list()
    selected_item = np.argmax(di2s)
    selected_items.append(selected_item)
    while len(selected_items) < max_length:
        k = len(selected_items) - 1
        ci_optimal = cis[:k, selected_item]
        di_optimal = math.sqrt(max(0, di2s[selected_item]))
        if di_optimal == 0:
            di_optimal = epsilon
        # start_x_time = time.time()
        elements = kernel_matrix[selected_item, :]
        # print("elements: ", time.time() - start_x_time)
        eis = (elements - np.dot(ci_optimal, cis[:k, :])) / di_optimal
        cis[k, :] = eis
        di2s -= np.square(eis)
        di2s[selected_item] = -np.inf
        selected_item = np.argmax(di2s)
        # if di2s[selected_item] < epsilon:
            # break
        selected_items.append(selected_item)
 
    S = np.sort(np.array(selected_items))
    return S, np.linalg.det(kernel_matrix[S.reshape(-1, 1), S.reshape(1, -1)]) 

def map_inference_dpp_local_search_2(L, k, verbose=False):
    start_time = time.time()
    greedy_sol, greedy_prob = map_inference_dpp_greedy(L, k)
    greedy_time = time.time() - start_time

    if verbose:
        print("Prob: ", greedy_prob)

    cur_sol = greedy_sol.copy()
    cur_prob = greedy_prob
    obj_greedy = greedy_prob

    N = L.shape[0]
    all_idx = np.array(range(N))
    ns_idx = np.setdiff1d(all_idx, cur_sol)

    # L = np.arange(100).reshape(10, 10)
    L_S = L[cur_sol[:, np.newaxis], cur_sol]
    it = 0

    while True:
        start_iter_time = time.time()

        idx = np.array(range(len(cur_sol)))
        best_removal_idx = 0
        best_removal_prob = 0

        for i in range(len(cur_sol)):
            # cur_sol[i], cur_sol[-1] = cur_sol[-1], cur_sol[i]
            idx[i], idx[-1] = idx[-1], idx[i]
            L_Se = L_S[idx[:-1, np.newaxis], idx[:-1]]
            prob = np.linalg.det(L_Se)

            if prob > best_removal_prob:
                best_removal_idx = i
                best_removal_prob = prob
        obj_loc = best_removal_prob

        brid = best_removal_idx
        br = cur_sol[brid]

        best_neighbors = cur_sol.copy()
        best_add = -1
        best_neighbors_prob = cur_prob
        localopt = True

        for v in ns_idx:
            cur_sol[brid] = v
            L_S[brid, :] = L[v, cur_sol]
            L_S[:, brid] = L[cur_sol, v]
            prob = np.linalg.det(L_S)

            if prob > best_neighbors_prob:
                best_neighbors_prob = prob
                best_add = v
                localopt = False

        if verbose:
            print("Iter {}:".format(it))
            print("remove item: ", br)
            print("add item: ", best_add)
            print("best_neighbors_prob: ", best_neighbors_prob)

        if not localopt:
            cur_sol[brid] = best_add
            cur_prob = best_neighbors_prob
            L_S[brid, :] = L[best_add, cur_sol]
            L_S[:, brid] = L[cur_sol, best_add]
            ns_idx = np.setdiff1d(all_idx, cur_sol)
        else:
            cur_sol = best_neighbors
            cur_prob = best_neighbors_prob
            break
        it += 1

    ls_time = time.time() - start_time
    return cur_sol, obj_loc, ls_time, greedy_sol, greedy_prob, greedy_time

def dpp_batches_without_replacement(
    X: np.ndarray,
    k: int,
    verbose: bool = False,
    normalize_for_cosine: bool = False,
):
    """
    Repeatedly sample size-k DPP subsets (without replacement) from X using your
    `map_inference_dpp_local_search_2(L, k, verbose=...)` routine, where L = X X^T (or cosine Gram).

    Returns
    -------
    batches : List[np.ndarray]
        Each element is an array of ORIGINAL indices of length k (one DPP batch).
    remainder : np.ndarray
        ORIGINAL indices left over (< k), not sampled by DPP.
    """
    # Work on a view that we shrink; keep a mapping to original indices
    remaining_idx = np.arange(X.shape[0])
    X_work = X.copy()

    batches = []

    while X_work.shape[0] >= k:
        # Optional cosine Gram (normalize rows) or plain dot-product Gram
        if normalize_for_cosine:
            norms = np.linalg.norm(X_work, axis=1, keepdims=True) + 1e-12
            Xn = X_work / norms
            L = Xn @ Xn.T
        else:
            L = X_work @ X_work.T

        # Your DPP MAP/local-search call (expects a PSD L and integer k)
        cur_sol, obj_loc, ls_time, greedy_sol, greedy_prob, greedy_time = \
            map_inference_dpp_local_search_2(L, k, verbose=verbose)

        # Make sure we have an integer 1D array of selected *current* indices
        cur_sol = np.asarray(cur_sol, dtype=int).ravel()
        # Map back to ORIGINAL indices and save this batch
        batch_orig = remaining_idx[cur_sol]
        batches.append(batch_orig)

        # Remove selected points from the working set
        mask = np.ones(X_work.shape[0], dtype=bool)
        mask[cur_sol] = False
        X_work = X_work[mask]
        remaining_idx = remaining_idx[mask]

        if verbose:
            print(f"[DPP] picked {len(cur_sol)}; remaining {X_work.shape[0]}")

    # Whatever is left (< k) — return their ORIGINAL indices as remainder
    remainder = remaining_idx.copy()
    return batches, remainder

In [ ]:
k = 256
dpp_batches, dpp_remainder = dpp_batches_without_replacement(embeddings, k, verbose=True)
print("Remainder size:", dpp_remainder.shape[0])
dpp_indices_list = [idx for batch in dpp_batches for idx in batch] + list(range(17152, 17398))
np.save("/home/hieunt/verl/data/organised_indices/dpp_indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy", np.array(dpp_indices_list))
len(dpp_indices_list)

Remainder size: 246


17398

In [ ]:

dists = []
mean_dists = []
for j in range(len(dpp_batches) - 1):
    for i in range(256):
        d = np.linalg.norm(embeddings[dpp_batches[j][i]] - embeddings[dpp_batches[j + 1]], axis=1)
        dists.append(d.min())
sum(dists) / len(dists)

np.float32(0.76710707)

In [103]:
import datasets 
parquet_path = "/home/hieunt/verl/data/fixprompt-dapo-math-17k.parquet"
ds = datasets.load_dataset("parquet", data_files=[parquet_path])["train"]
kmeans_indices = np.load("/home/hieunt/verl/data/organised_indices/kmeans_mst_indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy")
dpp_indices = np.load("/home/hieunt/verl/data/organised_indices/dpp_indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy")


In [104]:
# Reorder ds according to kmeans and dpp indices, then save
import pandas as pd

# Convert HuggingFace dataset to pandas DataFrame
if hasattr(ds, 'to_pandas'):
    df = ds.to_pandas()
else:
    df = pd.DataFrame(ds)

# Reorder by kmeans indices
kmeans_df = df.iloc[kmeans_indices].reset_index(drop=True)
kmeans_path = "/home/hieunt/verl/data/fixprompt-dapo-math-17k.kmeans_ordered.parquet"
kmeans_df.to_parquet(kmeans_path)
print(f"Saved kmeans-ordered dataset to {kmeans_path}")

# Reorder by dpp indices
dpp_df = df.iloc[dpp_indices].reset_index(drop=True)
dpp_path = "/home/hieunt/verl/data/fixprompt-dapo-math-17k.dpp_ordered.parquet"
dpp_df.to_parquet(dpp_path)
print(f"Saved dpp-ordered dataset to {dpp_path}")

Saved kmeans-ordered dataset to /home/hieunt/verl/data/fixprompt-dapo-math-17k.kmeans_ordered.parquet
Saved dpp-ordered dataset to /home/hieunt/verl/data/fixprompt-dapo-math-17k.dpp_ordered.parquet


In [110]:
new_ds = datasets.load_dataset("parquet", data_files=[kmeans_path])["train"]

Generating train split: 17398 examples [00:00, 182501.52 examples/s]
Generating train split: 17398 examples [00:00, 182501.52 examples/s]


In [106]:
new_ds

Dataset({
    features: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info', 'question'],
    num_rows: 17398
})

In [107]:
ds

Dataset({
    features: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info', 'question'],
    num_rows: 17398
})

In [ ]:
len(kmeans_indices), len(dpp_indices)

17398